# MiAge

[Youn and Wang (2018)](https://doi.org/10.1080/15592294.2017.1389361) infer mitotic age from 268 CpGs using a nonlinear methylation-transmission model. The authors' `function_library.r` and fitted `site_specific_parameters.Rdata` are preserved in [this pinned archive mirror](https://github.com/anilpsori/DNAm_pipelines_and_biomarkers/tree/d84d6adfc039355d5f9c962fc5faa7d1054f56b2/biomarkers/MiAge). We use those original files, and do not use the mirror's modified `MiAge.R` wrapper.

The original [Columbia download](https://www.columbia.edu/~sw2206/softwares/mitotic_age_R_code.zip) returned HTTP403 on 2026-10-02, so pristine ZIP identity could not be verified. All 268 CpG IDs, genes, chromosomes, coordinates and informativeness scores in the mirrored author table match the [original publisher supplement](https://doi.org/10.6084/m9.figshare.5620603.v2) in order. Native R executes the preserved author function to generate independent expected predictions.

The model minimizes `sum((c + b**(n-1)*d - beta)**2)` separately per sample using the authors' five L-BFGS-B starts and bounds of 10 to 10,000. The CPU float64 optimizer returns a tensor on the original device. Missing CpGs and supplied NaNs are excluded from the objective. With every beta missing, the original optimizer returns its first start, 2,008: this is not a biological estimate.

The author code/parameter archive declares no license. The separate publisher supplement is CC BY4.0; that does not establish a license for the software weights. Build-only Python dependency: `rdata`.

In [1]:
from pathlib import Path
import importlib.util
import tempfile
import os
import urllib.request
import pyaging as pya

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "clocks/build_replication.py").is_file())
spec = importlib.util.spec_from_file_location("build_replication", root / "clocks/build_replication.py")
builder = importlib.util.module_from_spec(spec)
spec.loader.exec_module(builder)
source = Path(os.environ.get("PYAGING_MIAGE_SOURCE", str(Path(tempfile.gettempdir()) / "MiAge-original-mirror")))
source.mkdir(exist_ok=True)
base = "https://raw.githubusercontent.com/anilpsori/DNAm_pipelines_and_biomarkers/" + builder.MIAGE_COMMIT + "/biomarkers/MiAge/"
for filename in builder.MIAGE_HASHES:
    if not (source / filename).exists():
        urllib.request.urlretrieve(base + filename, source / filename)
model = builder.build_miage(source)

In [2]:
model.metadata['clock_name'] = 'miage'
model.metadata['data_type'] = 'DNA methylation'  # Paper: DNA methylation measures
model.metadata['species'] = 'Homo sapiens'  # Paper: human tissue types
model.metadata['year'] = 2018
model.metadata['approved_by_author'] = '⌛'
model.metadata['citation'] = 'Youn, A., and Wang, S. (2018). The MiAge Calculator: a DNA methylation-based mitotic age calculator of human tissue types. Epigenetics, 13(2), 192–206.'
model.metadata['doi'] = 'https://doi.org/10.1080/15592294.2017.1389361'
model.metadata['notes'] = 'Mitotic-age estimator using the authors’ 268-CpG fitted b/c/d parameters and five-start bounded L-BFGS-B least squares (bounds of 10–10,000 on the relative-mitotic-age scale). The scale is identifiable only up to an unknown common factor, so scores are not absolute division counts. The scalar optimizer runs on CPU in float64, including when other clocks run on CUDA. Missing CpGs and supplied NaNs are omitted from each sample’s objective. The original all-NaN tie returns 2,008; this is an optimizer starting value with no biological meaning, so measured clock CpGs are required for interpretation. Original author function_library.r and site_specific_parameters.Rdata are preserved by an archived mirror at anilpsori/DNAm_pipelines_and_biomarkers commit d84d6adfc039355d5f9c962fc5faa7d1054f56b2; the original Columbia download returned HTTP403 during verification. All 268 CpG IDs and shared annotation fields match the original publisher supplement in order. No software/weight license is declared by the archived author files; the separate publisher supplement is CC BY4.0.'
model.metadata['research_only'] = None
model.metadata['tissue'] = ['multi-tissue']  # Paper: 4,020 tumor and adjacent normal tissue samples from eight TCGA cancer types
model.metadata['predicts'] = ['mitotic age']  # Paper: total number of lifetime cell divisions
model.metadata['training_target'] = ['replicative history']  # Paper: stochastic replication errors accumulated in the epigenetic inheritance process during cell divisions
model.metadata['unit'] = ['relative cell divisions']  # Paper: should be interpreted as a relative number rather than an actual number
model.metadata['model_type'] = 'dynamic methylation transmission model'  # Paper: the stochastic replication errors accumulated in the epigenetic inheritance process during cell divisions
model.metadata['platform'] = ['Illumina 450K']  # Paper: Illumina HumanMethylation450 BeadChip
model.metadata['population'] = 'human, age unspecified'  # Paper: 4,020 tumor and adjacent normal tissue samples
model.metadata['journal'] = 'Epigenetics'
model.metadata['last_author'] = 'Shuang Wang'
model.metadata['n_features'] = 268
model.metadata['citations'] = 98
model.metadata['citations_date'] = '2026-10-02'
model.version = "0.5.4"

`tests/data/replication/generate_miage_references.R` executes the authors' original inference function. Fixtures cover generative methylation at multiple mitotic ages, both bound-saturating directions, six random samples, partial missing values, absent probes and the original all-missing tie. `uv run pytest tests/models/test_replication.py` compares pipeline predictions across multiple batch sizes.

In [3]:
builder.save_models([model], root / "clocks/weights", root / "clocks/metadata/miage_0.5.4.pt")
print(model.metadata["clock_name"], len(model.features))

miage 268
